# Ingénierie des variables et imputation sans fuite

Ce notebook crée deux copies transformées des fichiers de données locaux : `data/data_config_1.csv` et `data/test_config_1.csv`. Les fichiers `train.csv` et `test.csv` restent inchangés. Les statistiques d’imputation sont apprises sur le train seulement; pendant la validation, chaque pli possède son propre ajustement.

Les CSV exportés facilitent l’inspection des variables transformées. Ils ne doivent pas servir d’entrée à une validation croisée, car leurs imputations ont été calculées sur tout le train. L’entraînement doit repartir de `train.csv` et `test.csv` : la pipeline appelle le même transformeur à l’intérieur de chaque pli.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

RACINE = Path.cwd()
if not (RACINE / "config.json").is_file():
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

from src.config import load_config
from src.data import load_competition_data
from src.preprocessing import FeatureEngineeringTransformer

config = load_config(RACINE / "config.json")
donnees = load_competition_data(config)
print(f"Train brut : {donnees.features.shape}; test brut : {donnees.test_features.shape}")
print(f"Dossier des données : {config.data_dir.resolve()}")

C:\Documents\Cours\M2\P1\Supervise\App_sup_M2\src\data.py:50: DtypeWarning: Columns (34: co-applicant_ethnicity-1, 38: co-applicant_race-1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(
C:\Documents\Cours\M2\P1\Supervise\App_sup_M2\src\data.py:50: DtypeWarning: Columns (34: co-applicant_ethnicity-1, 38: co-applicant_race-1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(


Train brut : (105000, 56); test brut : (45000, 56)
Dossier des données : C:\Documents\Cours\M2\P1\Supervise\App_sup_M2\data


## 1. État des valeurs manquantes et décisions

Les champs de catégorie peuvent représenter les valeurs manquantes par une chaîne vide. Le tableau ci-dessous compte les chaînes vides et les valeurs manquantes pandas. `applicant_age` est imputé par le mode même si son taux dépasse légèrement 2 %, conformément au choix explicite pour cette variable.

In [2]:
manquants = donnees.features.isna() | donnees.features.astype("string").apply(
    lambda colonne: colonne.str.strip().eq("")
)
resume_manquants = pd.DataFrame({
    "manquants": manquants.sum(),
    "taux_%": (manquants.mean() * 100).round(3),
})
resume_manquants = resume_manquants[resume_manquants["manquants"] > 0].sort_values(
    "taux_%", ascending=False
)
display(resume_manquants)

ltv_observe = pd.to_numeric(
    donnees.features["loan_to_value_ratio"].replace("", pd.NA), errors="coerce"
).dropna()
print(
    f"LTV observé : moyenne = {ltv_observe.mean():,.2f} %, "
    f"médiane = {ltv_observe.median():,.2f} %, "
    f"99e percentile = {ltv_observe.quantile(.99):,.2f} %"
)
for colonne in ("property_value", "income"):
    valeurs = pd.to_numeric(donnees.features[colonne], errors="coerce").dropna()
    print(f"{colonne} : moyenne = {valeurs.mean():,.2f}; médiane = {valeurs.median():,.2f}")
print(f"Revenus observés négatifs conservés : {(pd.to_numeric(donnees.features['income'], errors='coerce') < 0).sum()}")

,manquants,taux_%
co-applicant_age,58693,55.898
loan_to_value_ratio,11874,11.309
debt_to_income_ratio,7679,7.313
property_value,4756,4.53
income,4578,4.36
loan_term,3307,3.15
applicant_age,3088,2.941
county_code,1339,1.275
state_code,824,0.785
total_units,437,0.416


LTV observé : moyenne = 6,528.79 %, médiane = 90.00 %, 99e percentile = 104.72 %
property_value : moyenne = 515,903.69; médiane = 355,000.00
income : moyenne = 154.18; médiane = 107.00
Revenus observés négatifs conservés : 28


### Règles retenues

- Supprimer `co-applicant_age`, qui présente beaucoup de valeurs manquantes et n’est pas retenue.
- Classer le LTV en `≤80`, `>80–100` et `>100`. Les bornes 80 et 100 appartiennent respectivement aux deux premières tranches. Pour imputer un LTV manquant, calculer sur le train courant le 99e percentile puis la moyenne des valeurs observées plafonnées à ce percentile. Le plafonnement ne change pas les LTV observés; ceux-ci sont classés à partir de leur valeur réelle.
- Regrouper les catégories DTI de 20 % à 49 % dans `20%-49%`, puis imputer les manquants par le mode après regroupement.
- Imputer `applicant_age` par le mode.
- Imputer `property_value` et `income` par leur médiane. La médiane résiste mieux aux longues queues de distribution. Les valeurs observées, dont les 28 revenus négatifs, restent intactes.
- Imputer par le mode les codes de comté et d’État, `total_units`, `conforming_loan_limit` et les codes de race/ethnicité. Le mode convient aux codes catégoriels et au compte discret `total_units`.
- Remplacer les valeurs manquantes de `loan_term` par 330.

Le transformeur commun implémente ces règles. Les médianes, modes et statistiques LTV sont ajustés sur le train du pli en validation, et sur le train complet pour produire les exports.

## 2. Vérifier le calcul local à chaque pli

Cette vérification garde les données transformées des plis en mémoire et n’écrit aucun CSV intermédiaire. Le jeu de validation de chaque pli reçoit les statistiques apprises sur les autres plis seulement.

In [3]:
plis = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.master_seed)
resume_plis = []

for numero, (indices_train, indices_validation) in enumerate(
    plis.split(donnees.features, donnees.target), start=1
):
    X_train_pli = donnees.features.iloc[indices_train]
    X_validation_pli = donnees.features.iloc[indices_validation]
    transformeur_pli = FeatureEngineeringTransformer().fit(X_train_pli)
    train_pli = transformeur_pli.transform(X_train_pli)
    validation_pli = transformeur_pli.transform(X_validation_pli)

    assert list(train_pli.columns) == list(validation_pli.columns)
    assert "co-applicant_age" not in train_pli.columns
    assert not train_pli.isna().any().any()
    assert not validation_pli.isna().any().any()
    resume_plis.append({
        "pli": numero,
        "LTV p99": transformeur_pli.ltv_cap_,
        "moyenne LTV plafonnée": transformeur_pli.ltv_mean_,
        "médiane property_value": transformeur_pli.numeric_medians_["property_value"],
        "médiane income": transformeur_pli.numeric_medians_["income"],
        "mode DTI regroupé": transformeur_pli.mode_imputations_["debt_to_income_ratio"],
        "mode applicant_age": transformeur_pli.mode_imputations_["applicant_age"],
        "lignes train / validation": f"{len(train_pli):,} / {len(validation_pli):,}",
    })

display(pd.DataFrame(resume_plis).set_index("pli").round(3))

,LTV p99,moyenne LTV plafonnée,médiane property_value,médiane income,mode DTI regroupé,mode applicant_age,lignes train / validation
pli,,,,,,,
1,104.690,83.613,355000.0,107.0,20%-49%,25-34,"84,000 / 21,000"
2,104.804,83.667,355000.0,107.0,20%-49%,25-34,"84,000 / 21,000"
3,104.700,83.671,355000.0,107.0,20%-49%,25-34,"84,000 / 21,000"
4,104.760,83.692,355000.0,107.0,20%-49%,25-34,"84,000 / 21,000"
5,104.674,83.575,355000.0,107.0,20%-49%,25-34,"84,000 / 21,000"


## 3. Apprendre sur tout le train et créer les deux exports

Pour les exports de consultation, on ajuste une dernière fois les statistiques sur tout le train. Le train transformé conserve `id` et la cible; le test conserve `id` et n’ajoute pas de cible. Les variables explicatives gardent le même ordre dans les deux fichiers.

In [4]:
fichiers_bruts = [config.data_dir / "train.csv", config.data_dir / "test.csv"]
empreintes_avant = {
    chemin.name: hashlib.sha256(chemin.read_bytes()).hexdigest()
    for chemin in fichiers_bruts
}

transformeur_complet = FeatureEngineeringTransformer().fit(donnees.features)
train_transforme = transformeur_complet.transform(donnees.features).reset_index(drop=True)
test_transforme = transformeur_complet.transform(donnees.test_features).reset_index(drop=True)

colonnes_variables = list(train_transforme.columns)
assert colonnes_variables == list(test_transforme.columns)
assert "co-applicant_age" not in colonnes_variables
assert not train_transforme.isna().any().any()
assert not test_transforme.isna().any().any()

ltv_brut = pd.to_numeric(
    donnees.features["loan_to_value_ratio"].replace("", pd.NA), errors="coerce"
).reset_index(drop=True)
ltv_observe = ltv_brut.notna()
classes_ltv_observees = pd.cut(
    ltv_brut[ltv_observe],
    bins=[-np.inf, 80, 100, np.inf],
    labels=["≤80", ">80–100", ">100"],
).astype(object)
assert train_transforme.loc[ltv_observe, "loan_to_value_ratio"].tolist() == list(classes_ltv_observees)
revenu_brut = pd.to_numeric(donnees.features["income"], errors="coerce").reset_index(drop=True)
revenus_negatifs = revenu_brut < 0
assert train_transforme.loc[revenus_negatifs, "income"].tolist() == revenu_brut[revenus_negatifs].tolist()

train_export = pd.concat(
    [
        donnees.train_ids.rename(config.id_column).reset_index(drop=True),
        donnees.target.rename(config.target_column).reset_index(drop=True),
        train_transforme,
    ],
    axis=1,
)
test_export = pd.concat(
    [donnees.test_ids.rename(config.id_column).reset_index(drop=True), test_transforme],
    axis=1,
)
assert list(train_export.columns[2:]) == list(test_export.columns[1:])

chemin_train = config.data_dir / "data_config_1.csv"
chemin_test = config.data_dir / "test_config_1.csv"
train_export.to_csv(chemin_train, index=False)
test_export.to_csv(chemin_test, index=False)

empreintes_apres = {
    chemin.name: hashlib.sha256(chemin.read_bytes()).hexdigest()
    for chemin in fichiers_bruts
}
assert empreintes_avant == empreintes_apres
print(f"Train transformé : {chemin_train} — {train_export.shape}")
print(f"Test transformé : {chemin_test} — {test_export.shape}")
print("Les empreintes des fichiers train.csv et test.csv sont inchangées.")

Train transformé : C:\Documents\Cours\M2\P1\Supervise\App_sup_M2\data\data_config_1.csv — (105000, 57)
Test transformé : C:\Documents\Cours\M2\P1\Supervise\App_sup_M2\data\test_config_1.csv — (45000, 56)
Les empreintes des fichiers train.csv et test.csv sont inchangées.


## 4. Valeurs apprises et contrôle des sorties

Les paramètres ci-dessous proviennent du train complet. Ils servent uniquement à ces exports complets et au réentraînement final. En validation croisée, il faut laisser `build_model_pipeline` ajuster un nouveau transformeur à l’intérieur de chaque pli.

In [5]:
stats = transformeur_complet.statistics_
print(f"LTV : p99 = {stats['loan_to_value_ratio']['cap_p99']:.3f} %; "
      f"moyenne plafonnée = {stats['loan_to_value_ratio']['mean_after_cap']:.3f} %")
print(f"Médiane property_value : {stats['property_value_median']:,.2f}")
print(f"Médiane income : {stats['income_median']:,.2f}")
print(f"Valeur de remplacement loan_term : {stats['loan_term_fill']}")
display(pd.Series(stats["mode_imputations"], name="mode appris").to_frame())

display(train_export.head())
display(test_export.head())
print("Colonnes explicatives identiques et dans le même ordre :",
      list(train_export.columns[2:]) == list(test_export.columns[1:]))
print("Valeurs manquantes restantes — train :", int(train_export.isna().sum().sum()))
print("Valeurs manquantes restantes — test :", int(test_export.isna().sum().sum()))

LTV : p99 = 104.719 %; moyenne plafonnée = 83.644 %
Médiane property_value : 355,000.00
Médiane income : 107.00
Valeur de remplacement loan_term : 330


,mode appris
county_code,4013.0
state_code,TX
total_units,1.0
conforming_loan_limit,C
applicant_ethnicity-1,2.0
applicant_race-1,5.0
co-applicant_ethnicity-1,5.0
co-applicant_race-1,8.0
applicant_age,25-34
debt_to_income_ratio,20%-49%


,id,accepted,loan_amount,loan_to_value_ratio,property_value,income,loan_term,total_units,state_code,derived_msa-md,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,544438,1,705000.0,≤80,1495000.0,655.0,360.0,1.0,CA,34900,...,1,1,1,4493,49.83,121100,78,1094,1486,55
1,316090,1,295000.0,≤80,555000.0,63.0,360.0,1.0,NC,99999,...,1,1,6,1604,13.22,71500,140,547,2188,39
2,153632,1,305000.0,≤80,385000.0,109.0,360.0,1.0,VA,40060,...,1,1,1,4780,40.92,110000,71,781,1527,53
3,400700,1,765000.0,≤80,1305000.0,190.0,360.0,1.0,NC,16740,...,1,1,1,7126,32.88,100300,273,1991,2094,0
4,205569,0,1775000.0,≤80,2955000.0,15675.0,360.0,1.0,FL,33124,...,1,1,1,6006,53.48,79400,167,1043,723,56


,id,loan_amount,loan_to_value_ratio,property_value,income,loan_term,total_units,state_code,derived_msa-md,county_code,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,923582,425000.0,≤80,625000.0,87.0,360.0,1.0,VA,11694,51059.0,...,1,1,1,7696,77.17,164200,98,1505,1645,17
1,801040,275000.0,>80–100,285000.0,105.0,360.0,1.0,PA,25420,42043.0,...,1,1,1,7108,43.85,97900,77,1464,1902,52
2,705186,155000.0,>80–100,165000.0,51.0,300.0,1.0,NC,99999,37083.0,...,1,1,6,3813,59.01,71500,70,964,1797,38
3,614701,105000.0,>80–100,105000.0,26.0,360.0,2.0,NY,99999,36057.0,...,1,1,1,3189,7.12,83800,104,956,1348,42
4,951449,165000.0,>80–100,175000.0,74.0,360.0,1.0,OK,46140,40113.0,...,1,1,3,5403,87.38,85300,67,1169,2360,45


Colonnes explicatives identiques et dans le même ordre : True
Valeurs manquantes restantes — train : 0
Valeurs manquantes restantes — test : 0
